# Part1: Langchain 기초

## 1. 환경 구성


### 1) 라이브러리 설치

In [ ]:
!pip install -q langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.3 MB/s eta 0:00:00


### 2) OpenGroq 인증키

In [ ]:
from google.colab import userdata
import os

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

## 2. LLM Chain

### 1) Prompt + LLM

- OpenAI 계열: gpt-, o1-, o3- 같은 이름은 워낙 유명하고OpenAI 고유 네이밍이라, model_provider 안 적어도 알아서 인식되는 경우가 많음.
- Gemini 계열: 같은 모델 이름이 Vertex AI 경로로도, Google GenAI 경로로도 로드될 수 있어서 명시 필수
- Groq 계열: llama-3.3-70b-versatile 같은 오픈소스 모델은 Groq 전용 이름이 아니라 Together AI, HuggingFace 등 여러 곳에서 같은 이름으로 서빙될 수 있어서, 이것도 명시 필요.
무료 티어 한도: 분당 30 요청, 분당 6,000 토큰, 일일 14,400 요청 정도 (모델마다 조금씩 다름)

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

result = llm.invoke("지구 자전의 주기는?")

In [ ]:
result.content

'23시간 56분 4.09초'

ChatPromptTemplate.from_template() 메소드는 문자열 형태의 템플릿을 인자로 받아, 해당 형식에 맞는 프롬프트 객체를 생성합니다. 여기서 사용된 템플릿 문자열은 "You are an expert in astronomy.  Answer the question in Korean only. : {input}"으로, 이는 모델에게 천문학 분야의 전문가로서 행동하라고 지시하고, {input} 부분에서 실제 질문을 받아 답변하도록 요청합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("You are an expert in astronomy. Answer the question in Korean only. : {input}")
prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='You are an expert in astronomy. Answer the question in Korean only. : {input}'), additional_kwargs={})])

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile")

# chain 연결
chain = prompt | llm

# chain 호출
chain.invoke({"input": "지구의 자전 주기는?"})

AIMessage(content='지구의 자전 주기는 23시간 56분 4초입니다. 이는 지구가 자전축을 기준으로 한 회전을 완료하는 데 걸리는 시간으로, 1일로 알려져 있습니다.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 58, 'total_tokens': 108, 'completion_time': 0.204033985, 'completion_tokens_details': None, 'prompt_time': 0.005239086, 'prompt_tokens_details': None, 'queue_time': 0.161260381, 'total_time': 0.209273071}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4545-d8f7-7ad1-aa58-9810ed021a8f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 50, 'total_tokens': 108})

 prompt 객체에 {"input": "지구의 자전 주기는?"} 라는 입력 값이 주어졌을 때, {input} 위치로 "지구의 자전 주기는?" 값이 전달되어 질문에 대한 프롬프트를 완성합니다. 완성된 프롬프트는 그 후 LLM에 전달되어, 모델이 입력된 질문에 대한 답변을 생성하게 됩니다. 모델의 답변은 인공지능 모델의 메시지를 나타내는 AIMessage 객체로 제공됩니다.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("You are an expert in astronomy. Answer the question in Korean only. : {input}")
llm = ChatGroq(model="llama-3.3-70b-versatile")
output_parser = StrOutputParser()

# LCEL(LangChain Expression Language) chaining
chain = prompt | llm | output_parser

# chain 호출
chain.invoke({"input": "지구의 자전 주기는?"})

'지구의 자전 주기는 23시간 56분 4초입니다. 이 시간은 지구가 자전하여 한 번 회전하는데 걸리는 시간이며, 24시간으로 기준하는 일반적인 시간과 약간 다릅니다.'

### 2) Multiple Chains

멀티 체인(Multi-Chain)은 여러 개의 체인을 연결하거나 병렬로 실행하여 복잡한 워크플로우를 구성하는 패턴입니다. LCEL(LangChain Expression Language)을 사용하면 체인을 파이프(|)와 RunnableParallel로 유연하게 조합할 수 있습니다.
- 단일 체인 = 파이프 하나로 쭉 (prompt → llm → parser)
- 멀티체인 = 체인의 결과를 다른 체인의 재료로 씀 (체인 A의 출력 → 체인 B의 입력)

In [ ]:
prompt1 = ChatPromptTemplate.from_template("translates {korean_word} to English")
prompt2 = ChatPromptTemplate.from_template(
    "explain {english_word} using oxford dictionary definition."
     "Translate your entire answer into Korean. Do not repeat the same sentence in different words."
     "Do not use any other language, characters, or scripts."
)

llm = ChatGroq(model="llama-3.3-70b-versatile")

chain1 = prompt1 | llm | StrOutputParser()

chain1.invoke({"korean_word": "미래"})

'"미래" (mi-rae) translates to "future" in English.'

In [ ]:
chain2 = (
    {"english_word": chain1}
    | prompt2
    |llm
    | StrOutputParser()
)

chain2.invoke({"korean_word": "미래"})

'미래는 시간의 흐름에 따라 현재보다 뒤에 있는 시기 또는 시대라는 뜻을 가진다. 시간의 연속성 속에서 현재와 과거를 거쳐 앞으로 나아가는 시점을 말한다. 따라서 미래는 현재의 선택과 행동이 영향을 미치는 결과물이기도 하다. 이러한 미래는 불확실성과 예측 불가능성을 내포한다. 시간의 흐름에 따라 상황과環境이 변하기 때문이다. 그러므로 미래에 대한 준비와 계획은 매우 중요하다. 이를 통해 개인과 사회는 앞으로 나아가며 발전할 수 있다.'

## 3. Prompt

### 1) PromptTemplate

.format()이 하는 일: 템플릿의 빈칸({name}, {age})에 값을 채워서 순수 문자열을 만들어줌
- .format() = "프롬프트 템플릿이 실제로 어떻게 채워지는지 미리 확인하고 싶을 때" 쓰는 디버깅/검증용 메서드
- .invoke() = "이 프롬프트를 실제로 LLM한테 보내서 답을 받고 싶을 때" 쓰는 실행 메서드

In [ ]:
from langchain_core.prompts import PromptTemplate

template_text = "안녕하세요, 제 이름은 {name}이고, 나이는 {age}살입니다."

# PromptTemplate 인스턴스 생성
prompt_template = PromptTemplate.from_template(template_text)

filled_prompt = prompt_template.format(name="홍길동", age=30)

filled_prompt

'안녕하세요, 제 이름은 홍길동이고, 나이는 30살입니다.'

PromptTemplate 클래스는 문자열을 기반으로 프롬프트 템플릿을 생성하고, + 연산자를 사용하여 직접 결합하는 동작을 지원

In [ ]:
# 문자열 템플릿 결합(PromptTemplate + PromptTemplate + 문자열)
combined_prompt = (
    prompt_template
    + PromptTemplate.from_template("\n\n아버지를 아버지라 부를 수 없습니다.")
    + "\n\n{language}로 번역해주세요."
)

combined_prompt

PromptTemplate(input_variables=['age', 'language', 'name'], input_types={}, partial_variables={}, template='안녕하세요, 제 이름은 {name}이고, 나이는 {age}살입니다.\n\n아버지를 아버지라 부를 수 없습니다.\n\n{language}로 번역해주세요.')

\n 하나(줄바꿈 1번)만 써도 되고, 아예 공백( )만 써도 동작은 해요 — 다만 여러 지시사항을 섞을 때는 \n\n(문단 구분)이 모델한테 더 명확한 신호를 줘서 흔히 쓰는 패턴

In [ ]:
combined_prompt.format(name="홍길동", age=30, language="한국어")

'안녕하세요, 제 이름은 홍길동이고, 나이는 30살입니다.\n\n아버지를 아버지라 부를 수 없습니다.\n\n한국어로 번역해주세요.'

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.3-70b-versatile")
chain = combined_prompt | llm | StrOutputParser()
chain.invoke({"age":30, "language":"영어", "name":"홍길동"})

'Hello, my name is Hong Gildong and I\'m 30 years old.\n\nI can\'t call my father, father.\n\n(However, a more poetic translation of the second sentence would be: \n\nI am a son who cannot call his father, father.)\n\nThis phrase is a famous opening line from the classic Korean novel "The Tale of Hong Gildong" by Heo Gyun. It sets the tone for the story, which explores themes of identity, morality, and social class.'

### 2) ChatPromptTemplate

#### ChatPromptTemplates + ChatModels ( 여러 메시지 입력 -> 단일 메시지 출력)
- 채팅 메시지를 원소로 갖는 리스트 형태
- 구성 : 각 채팅 메시지는 역할과 내용이 짝을 이루는 형태

In [ ]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "이 시스템은 천문학 질문에 답변할 수 있습니다."),
    ("user","{user_input}")
])

messages = chat_prompt.format_messages(user_input="태양계에서 가장 큰 해성은 무엇인가요?")
messages

[SystemMessage(content='이 시스템은 천문학 질문에 답변할 수 있습니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='태양계에서 가장 큰 해성은 무엇인가요?', additional_kwargs={}, response_metadata={})]

### Message 유형
- SystemMessage: 시스템의 기능을 설명합니다.
- HumanMessage: 사용자의 질문을 나타냅니다.
- AIMessage: AI 모델의 응답을 제공합니다.

In [ ]:
chain = chat_prompt | llm | StrOutputParser()

chain.invoke({"user_input": "태양계에서 가장 큰 행성은 무엇인가요?"})

'태양계에서 가장 큰 행성은 목성입니다. 목성은 태양계의 다섯 번째 행성으로, 지구의 약 11.2배의 크기와 약 318배의 질량을 가지고 있습니다. 목성은 주로 가스와 액체로 구성되어 있으며, 강력한 자기장과 대기 조건으로 인해 독특한 특징을 가지고 있습니다.'

### 3) MessagePromptTemplate

In [ ]:
# MessagePromptTemplate 활용

from langchain_core.prompts import SystemMessagePromptTemplate,  HumanMessagePromptTemplate

chat_prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("이 시스템은 천문학 질문에 답변할 수 있습니다."),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)


messages = chat_prompt.format_messages(user_input="태양계에서 가장 큰 행성은 무엇인가요?")
messages


[SystemMessage(content='이 시스템은 천문학 질문에 답변할 수 있습니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='태양계에서 가장 큰 행성은 무엇인가요?', additional_kwargs={}, response_metadata={})]

In [ ]:
chain = chat_prompt | llm | StrOutputParser()

chain.invoke({"user_input": "태양계에서 가장 큰 행성은 무엇인가요?"})

'태양계에서 가장 큰 행성은 목성입니다. 목성은 태양계의 다섯 번째 행성으로, 태양에서 가장 멀리 떨어져 있는 행성은 아니지만 크기와 질량에서 가장 큰 행성입니다. 목성의 직경은 약 14만 2천 킬로미터로, 지구의 직경의 약 11배이고, 질량은 약 1.9 x 10^27 킬로그램으로, 지구의 질량의 약 318배입니다.'

## 5. Model Parameter

### 1) 모델 클래스 유형

Groq은 OpenAI처럼 completion 스타일의 별도 LLM 클래스를 제공하지 않고, ChatGroq 하나만 제공 -> 실행은 못하고 참고용으로 OpenAI 코드 붙어넣음

#### LLM

텍스트 문자열을 입력으로 받아 처리한 후, 텍스트 문자열을 반환

In [ ]:
from langchain_openai import OpenAI

llm = OpenAI()

llm.invoke("한국의 대표적인 관광지 3군데를 추천해주세요.")

'\n\n1. 경복궁 (서울) - 대한민국의 대표적인 궁궐로, 조선 왕조의 건립 및 건축 양식을 보여주는 곳입니다. 아름다운 정원과 건물들이 많이 보존되어 있어서 한국의 역사와 문화를 체험할 수 있습니다.\n\n2. 제주도 (제주특별자치도) - 한국에서 가장 인기 있는 관광지로, 아름다운 자연경관과 독특한 문화, 맛집들이 많이 있는 곳입니다. 해변, 산, 화산, 동굴 등 다양한 자연 명소와 함께 제주 특유의 문화인 풍류와 제주 오일장 등을 즐길 수 있습니다.\n\n3. 부산 남포동 (부산) - 부산'

#### ChatModel

메시지의 리스트를 입력으로 받고, 하나의 메시지를 반환하며 이 모델은 대화형 상황에 최적화되어 있음.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "이 시스템은 여행 전문가입니다."),
    ("user", "{user_input}"),
])

chain = chat_prompt | chat
chain.invoke({"user_input": "안녕하세요? 한국의 대표적인 관광지 3군데를 추천해주세요."})

AIMessage(content='안녕하세요! 한국의 대표적인 관광지 3군데를 추천해드리겠습니다.\n\n1. 경복궁 (Gyeongbokgung Palace) - 서울의 중심에 위치한 경복궁은 조선 왕조의 궁궐로서 가장 크고 아름다운 궁궁 중 하나입니다. 경복궁에서는 한국 전통 건축물을 감상하고 궁중문화를 체험할 수 있습니다.\n\n2. 부산 해운대해수욕장 (Haeundae Beach) - 부산의 대표적인 해변으로 유명한 해운대해수욕장은 아름다운 모래사장과 시원한 바다를 즐길 수 있는 장소입니다. 해수욕뿐만 아니라 주변의 맛집과 관광명소도 풍부합니다.\n\n3. 경주 (Gyeongju) - 경주는 한국의 역사와 문화가 깃들어 있는 도시로서 세계문화유산에 등재된 고구려 유적지와 신라 왕궁, 불교 사찰 등이 있습니다. 경주는 한국의 역사적인 유산을 체험할 수 있는 멋진 관광지입니다.\n\n이렇게 한국의 대표적인 관광지 3군데를 추천해드렸습니다. 혹시 더 궁금한 사항이 있으시면 언제든지 물어보세요!')

### 2) 모델 파라미터 설정

frequency_penalty, presence_penalty는 OpenAI 전용 파라미터라서, Groq/Llama 모델에서는 아예 무시되거나 에러남.

#### **모델 생성 시점**에 모델에 직접 파라미터를 전달

In [ ]:
# OpenAI
from langchain_openai import ChatOpenAI

# 모델 파라미터 설정
params = {
    "temperature": 0.7,         # 생성된 텍스트의 다양성 조정
    "max_tokens": 100,          # 생성할 최대 토큰 수
}

kwargs = {
    "frequency_penalty": 0.5,   # 이미 등장한 단어의 재등장 확률
    "presence_penalty": 0.5,    # 새로운 단어의 도입을 장려
    "stop": ["\n"]              # 정지 시퀀스 설정

}

# 모델 인스턴스를 생성할 때 설정
model = ChatOpenAI(model="gpt-3.5-turbo-0125", **params, model_kwargs = kwargs)


# 모델 호출
question = "태양계에서 가장 큰 행성은 무엇인가요?"
response = model.invoke(input=question)

# 전체 응답 출력
print(response)

content='태양계에서 가장 큰 행성은 목성입니다. 목성은 태양계에서 가장 큰 행성으로, 지구의 약 11배 크기입니다. 목성은 가스 행성으로 주로 수소와 헬륨으로 구성되어 있습니다.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 48, 'total_tokens': 108, 'completion_time': 0.179206762, 'completion_tokens_details': None, 'prompt_time': 0.041053683, 'prompt_tokens_details': None, 'queue_time': 0.418653111, 'total_time': 0.220260445}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f45c8-cdb5-7673-9e81-8971ca59fe1f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 60, 'total_tokens': 108}


In [ ]:
#Groq
from langchain_groq import ChatGroq

# 1) 생성 시점 파라미터
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7, max_tokens=100)

question = "태양계에서 가장 큰 행성은 무엇인가요?"
response = model.invoke(question)
print(response.content)


목성은 태양계에서 가장 큰 행성입니다.


#### **모델 호출 시점** 에 파라미터 전달

In [ ]:
# OpenAI
# 모델 파라미터 설정
params = {
    "temperature": 0.7,
    "max_tokens": 10,
}

# 모델 인스턴스를 호출할 때 전달
response = model.invoke(input=question, **params)

# 전체 응답 출력
print(response)

In [ ]:
#Groq
# 2) 호출 시점 파라미터로 덮어쓰기
params = {"temperature": 0.7, "max_tokens": 10}
response2 = model.invoke(input=question, **params)
print(response2.content)

태양계에서 가장 큰 행성은 목


#### 모델에 추가적인 파라미터를 전달 (bind 메서드 사용)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_messages([
    ("system", "이 시스템은 천문학 질문에 답변할 수 있습니다."),
    ("user", "{user_input}"),
])

model = ChatGroq(model="llama-3.3-70b-versatile", max_tokens=100)

messages = prompt.format_messages(user_input="태양계에서 가장 큰 행성은 무엇인가요?")

before_answer = model.invoke(messages)
print(before_answer)

# bind 메서드로 호출 시점에 max_tokens 덮어쓰기
chain = prompt | model.bind(max_tokens=10)

after_answer = chain.invoke({"user_input": "태양계에서 가장 큰 행성은 무엇인가요?"})
print(after_answer)

content='태양계에서 가장 큰 행성은 목성입니다. 목성은 지구의 약 11배 크기이며, 태양계에서 가장 큰 행성으로 알려져 있습니다.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 61, 'total_tokens': 102, 'completion_time': 0.108608592, 'completion_tokens_details': None, 'prompt_time': 0.006145307, 'prompt_tokens_details': None, 'queue_time': 0.161201191, 'total_time': 0.114753899}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f45d4-3f99-7e52-afe7-1dfe6d528532-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 61, 'output_tokens': 41, 'total_tokens': 102}
content='태양계에서 가장 큰 행성은 목' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 61, 'total_tokens': 71, 'completion_time': 0.014528065, 'completion_tokens_details': None, 'prompt_time': 0.005656821, 'prompt_tokens_details': None,

## 6. Output Parsers

### 1) CSV Parser

### 2) JSON Parser